In [ ]:
# @title 🚀 INSTALLAZIONE BASE (Esegui UNA VOLTA)
# @markdown Installa uv, pacchetti Python e crea directory per i modelli

import os
import subprocess
import sys
from pathlib import Path

print("="*70)
print("🚀 Installazione Base - Multi-Model Quantizer")
print("="*70)

# ============================================
# INSTALLA UV
# ============================================
print("\n📦 Installazione uv package manager...")
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    print("✅ uv installato via pip")
except:
    print("   Tentativo installer standalone...")
    subprocess.run("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True)
    os.environ["PATH"] = f"{os.environ['HOME']}/.cargo/bin:" + os.environ["PATH"]

result = subprocess.run(["uv", "--version"], capture_output=True, text=True)
print(f"✅ uv version: {result.stdout.strip()}")

# ============================================
# CREA AMBIENTE UV
# ============================================
UV_PROJECT_DIR = "/content/gguf_project"
UV_VENV_DIR = f"{UV_PROJECT_DIR}/.venv"

print("\n🐍 Configurazione ambiente Python...")
os.makedirs(UV_PROJECT_DIR, exist_ok=True)
os.chdir(UV_PROJECT_DIR)

if not os.path.exists("pyproject.toml"):
    subprocess.run(["uv", "init", "--no-readme", "--no-workspace"], check=True)
    print("✅ uv project inizializzato")

if not os.path.exists(UV_VENV_DIR):
    subprocess.run(["uv", "venv"], check=True)
    print("✅ Virtual environment creato")

# ============================================
# INSTALLA PACCHETTI PYTHON
# ============================================
print("\n📦 Installazione pacchetti Python...")

python_packages = [
    "torch",
    "transformers",
    "accelerate",
    "huggingface_hub>=0.32.0",
    "gguf",
    "tqdm"
]

for pkg in python_packages:
    print(f"   Installazione {pkg}...")
    subprocess.run(["uv", "add", pkg], check=True, capture_output=True)

subprocess.run(["uv", "sync"], check=True, capture_output=True)
print("✅ Tutti i pacchetti Python installati")

# ============================================
# CREA DIRECTORY MODELLI
# ============================================
os.chdir("/content")
os.makedirs("/content/models", exist_ok=True)
print("✅ Directory /content/models creata")

# ============================================
# SALVA PATHS
# ============================================
with open("/content/uv_project_path.txt", "w") as f:
    f.write(UV_PROJECT_DIR)
with open("/content/venv_path.txt", "w") as f:
    f.write(UV_VENV_DIR)

# Determina path Python del venv
if os.name == 'nt':
    VENV_PYTHON = os.path.join(UV_VENV_DIR, "Scripts", "python.exe")
else:
    VENV_PYTHON = os.path.join(UV_VENV_DIR, "bin", "python")

with open("/content/venv_python.txt", "w") as f:
    f.write(VENV_PYTHON)

print("\n" + "="*70)
print("✅ Installazione Base Completata!")
print("="*70)
print(f"🐍 Environment: {UV_VENV_DIR}")
print(f"📁 Modelli salvati in: /content/models")
print("\n💡 Ora puoi eseguire le celle dei modelli!")

In [ ]:
# @title 🔐 HUGGING FACE TOKEN (solo per modelli privati/gated)
# @markdown Inserisci il token solo se vuoi scaricare modelli privati

import os

# @markdown **Token di lettura Hugging Face:**
hf_token = "" # @param {type:"string"}

# @markdown 💡 Lascia vuoto per modelli pubblici (es. google/gemma-2-2b-it, openai/whisper-tiny)

# Salva token
if hf_token.strip():
    with open("/content/hf_token.txt", "w") as f:
        f.write(hf_token.strip())
    print("✅ Token salvato")
else:
    if os.path.exists("/content/hf_token.txt"):
        os.remove("/content/hf_token.txt")
    print("⚠️ Nessun token fornito - solo modelli pubblici accessibili")

In [ ]:
# @title 🎯 QUANTIZZA MODELLO LLM (Gemma, Llama, Mistral...)
# @markdown Seleziona modello e metodo di quantizzazione

import os
import subprocess
import sys
import time
import shutil
import re
import tempfile
from pathlib import Path
from huggingface_hub import snapshot_download, HfApi

# ============================================
# CONFIGURAZIONE MODELLO
# ============================================

# @markdown ---
# @markdown ### 🤗 Modello Hugging Face
llm_model_url = "google/gemma-2-2b-it" # @param {type:"string"}
# @markdown Esempi: google/gemma-2-2b-it, meta-llama/Llama-3.2-1B, mistralai/Mistral-7B-v0.1

# @markdown ---
# @markdown ### 🔧 Metodo di Quantizzazione
llm_quant_method = "Q4_K_M" # @param ["Q2_K", "Q3_K_S", "Q3_K_M", "Q3_K_L", "Q4_0", "Q4_K_S", "Q4_K_M", "Q5_0", "Q5_K_S", "Q5_K_M", "Q6_K", "Q8_0", "F16", "IQ1_S", "IQ2_XXS", "IQ2_XS", "IQ2_S", "IQ3_XXS", "IQ3_S", "IQ4_NL", "IQ4_XS", "IQ1_M"]

# @markdown ---
# @markdown ### 📊 Imatrix (migliora qualità - opzionale)
use_imatrix = False # @param {type:"boolean"}
imatrix_url = "" # @param {type:"string"}
# @markdown 💡 URL file imatrix (HF o Drive). Vuoto = usa groups_merged.txt predefinito

# @markdown ---
# @markdown ### 📁 Output
output_filename = "" # @param {type:"string"}
# @markdown 💡 Vuoto = automatico: [nome_modello]-[quant].gguf

# ============================================
# FUNZIONI UTILITY
# ============================================

def get_hf_token():
    if os.path.exists("/content/hf_token.txt"):
        with open("/content/hf_token.txt", "r") as f:
            return f.read().strip()
    return None

def extract_model_id(url):
    url = str(url).strip()
    if "huggingface.co" in url:
        match = re.search(r'huggingface\.co/([^/]+/[^/]+)', url)
        if match:
            return match.group(1)
    url = re.sub(r'^https?://', '', url)
    return url

def download_model(model_url, cache_dir, token=None):
    print(f"\n📥 Download: {model_url}")

    if os.path.exists(cache_dir):
        print(f"   Rimozione cache esistente...")
        shutil.rmtree(cache_dir)

    # Determina pattern di download
    dl_patterns = ["*.md", "*.json", "*.model", "*.safetensors", "*.bin"]

    try:
        api = HfApi(token=token if token else None)
        repo_files = list(api.list_repo_tree(repo_id=model_url, recursive=True))

        has_safetensors = any(f.path.endswith(".safetensors") for f in repo_files)
        if has_safetensors:
            dl_patterns = ["*.safetensors"] + dl_patterns

        tokenizer_files = ["tokenizer.json", "tokenizer.model", "vocab.json", "merges.txt"]
        for tf in tokenizer_files:
            if any(f.path.endswith(tf) for f in repo_files):
                dl_patterns.append(tf)
    except:
        dl_patterns = ["*.safetensors", "*.bin", "tokenizer.json", "tokenizer.model"]

    # Download
    snapshot_download(
        repo_id=model_url,
        local_dir=cache_dir,
        local_dir_use_symlinks=False,
        resume_download=True,
        allow_patterns=dl_patterns,
        token=token if token else None
    )

    print("✅ Download completato!")
    return cache_dir

def compile_llama_cpp():
    """Compila llama.cpp se non esiste"""
    os.chdir("/content")

    if os.path.exists("/content/llama.cpp/llama-quantize"):
        print("✅ llama.cpp già compilato")
        return True

    print("\n🔨 Compilazione llama.cpp in corso...")

    if not os.path.exists("llama.cpp"):
        subprocess.run("git clone https://github.com/ggml-org/llama.cpp", shell=True)

    os.chdir("llama.cpp")
    subprocess.run("make -j4", shell=True, capture_output=True)
    os.chdir("/content")

    if os.path.exists("/content/llama.cpp/llama-quantize"):
        print("✅ llama.cpp compilato con successo!")
        return True
    else:
        print("❌ Errore compilazione llama.cpp")
        return False

def download_imatrix_file(url):
    """Scarica file imatrix da URL"""
    if not url or not url.strip():
        return "/content/llama.cpp/groups_merged.txt"

    url = url.strip()
    output_path = "/content/imatrix_custom.txt"

    if "huggingface.co" in url:
        print(f"   Download imatrix da Hugging Face...")
        try:
            import requests
            response = requests.get(url)
            with open(output_path, "wb") as f:
                f.write(response.content)
            return output_path
        except Exception as e:
            print(f"   ⚠️ Errore download: {e}, uso default")
            return "/content/llama.cpp/groups_merged.txt"

    elif "drive.google.com" in url:
        print(f"   Download imatrix da Google Drive...")
        try:
            import gdown
            gdown.download(url, output_path, quiet=False)
            return output_path
        except:
            print(f"   ⚠️ Errore download Drive, uso default")
            return "/content/llama.cpp/groups_merged.txt"

    return "/content/llama.cpp/groups_merged.txt"

def generate_imatrix(model_path, imatrix_file, output_path):
    """Genera imatrix per il modello"""
    print(f"\n📊 Generazione importance matrix...")

    cmd = [
        "/content/llama.cpp/llama-imatrix",
        "-m", model_path,
        "-f", imatrix_file,
        "-ngl", "99",
        "-o", output_path
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"   ⚠️ Warning imatrix: {result.stderr[:200]}")
        return False

    print(f"✅ Imatrix generata: {output_path}")
    return True

def convert_llm_to_gguf(model_path, output_path, venv_python):
    """Converti modello HF in GGUF FP16"""
    print(f"\n🔄 Conversione a FP16 GGUF...")

    os.chdir("/content")
    convert_script = "/content/llama.cpp/convert_hf_to_gguf.py"

    cmd = [venv_python, convert_script, model_path, "--outfile", output_path, "--outtype", "f16"]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"❌ Errore conversione: {result.stderr}")
        return False

    if not os.path.exists(output_path):
        print(f"❌ File non creato: {output_path}")
        return False

    size = os.path.getsize(output_path) / (1024**3)
    print(f"✅ Conversione completata ({size:.2f} GB)")
    return True

def quantize_llm(model_path, quant_method, output_path, imatrix_path=None):
    """Quantizza modello GGUF"""
    print(f"\n🔧 Quantizzazione a {quant_method}...")

    if quant_method == "F16":
        shutil.copy(model_path, output_path)
        print("✅ Copiato (F16 senza compressione)")
        return True

    if imatrix_path and os.path.exists(imatrix_path):
        cmd = [
            "/content/llama.cpp/llama-quantize",
            "--imatrix", imatrix_path,
            model_path, output_path, quant_method
        ]
    else:
        cmd = [
            "/content/llama.cpp/llama-quantize",
            model_path, output_path, quant_method
        ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"❌ Errore quantizzazione: {result.stderr}")
        return False

    size = os.path.getsize(output_path) / (1024**3)
    print(f"✅ Quantizzazione completata ({size:.2f} GB)")
    return True

# ============================================
# MAIN ESECUZIONE
# ============================================

print("="*70)
print("🎯 Quantizzazione Modello LLM")
print("="*70)

start_time = time.time()

# Carica paths
with open("/content/venv_python.txt", "r") as f:
    VENV_PYTHON = f.read().strip()

model_url = extract_model_id(llm_model_url)
model_name = model_url.split('/')[-1]

print(f"📁 Modello: {model_url}")
print(f"🔧 Quantizzazione: {llm_quant_method}")
if use_imatrix:
    print(f"📊 Imatrix: Attivata")
print("="*70)

# 1. Compila llama.cpp se necessario
if not compile_llama_cpp():
    print("❌ Impossibile procedere senza llama.cpp")
    sys.exit(1)

# 2. Download modello
token = get_hf_token()
with tempfile.TemporaryDirectory(dir="/content") as download_dir:
    model_cache = os.path.join(download_dir, model_name)
    download_model(model_url, model_cache, token)

    # 3. Converti a GGUF FP16
    fp16_path = os.path.join(download_dir, f"{model_name}.fp16.gguf")
    if not convert_llm_to_gguf(model_cache, fp16_path, VENV_PYTHON):
        sys.exit(1)

    # 4. Imatrix se richiesto
    imatrix_data = None
    if use_imatrix:
        imatrix_file = download_imatrix_file(imatrix_url)
        imatrix_data = os.path.join(download_dir, "imatrix.dat")
        if not generate_imatrix(fp16_path, imatrix_file, imatrix_data):
            imatrix_data = None

    # 5. Quantizza
    if output_filename.strip():
        quantized_name = output_filename.strip()
        if not quantized_name.endswith(".gguf"):
            quantized_name += ".gguf"
    else:
        if use_imatrix:
            quantized_name = f"{model_name}-{llm_quant_method}-imat.gguf"
        else:
            quantized_name = f"{model_name}-{llm_quant_method}.gguf"

    quantized_path = os.path.join(download_dir, quantized_name)

    if not quantize_llm(fp16_path, llm_quant_method, quantized_path, imatrix_data):
        sys.exit(1)

    # 6. Copia in /content/models
    final_path = f"/content/models/{quantized_name}"
    shutil.copy2(quantized_path, final_path)

elapsed = time.time() - start_time

print("\n" + "="*70)
print("✅ QUANTIZZAZIONE COMPLETATA!")
print("="*70)
print(f"📁 File: {final_path}")
print(f"💾 Dimensione: {os.path.getsize(final_path) / (1024**3):.2f} GB")
print(f"⏱️ Tempo: {elapsed/60:.2f} minuti")
print("="*70)

In [ ]:
# @title 🎙️ QUANTIZZA MODELLO WHISPER
# @markdown Seleziona modello Whisper e metodo di quantizzazione

import os
import subprocess
import sys
import time
import shutil
import re
import tempfile
from pathlib import Path
from huggingface_hub import snapshot_download, HfApi

# ============================================
# CONFIGURAZIONE MODELLO
# ============================================

# @markdown ---
# @markdown ### 🤗 Modello Hugging Face
whisper_model_url = "openai/whisper-tiny" # @param {type:"string"}
# @markdown Esempi: openai/whisper-tiny, openai/whisper-base, openai/whisper-small, openai/whisper-medium, openai/whisper-large, openai/whisper-large-v3

# @markdown ---
# @markdown ### 🔧 Metodo di Quantizzazione
whisper_quant_method = "q5_0" # @param ["q2_0", "q2_1", "q3_0", "q3_1", "q4_0", "q4_1", "q5_0", "q5_1", "q8_0"]

# @markdown ---
# @markdown ### 📁 Output
output_filename = "" # @param {type:"string"}
# @markdown 💡 Vuoto = automatico: [nome_modello]-[quant].bin

# ============================================
# FUNZIONI UTILITY
# ============================================

def get_hf_token():
    if os.path.exists("/content/hf_token.txt"):
        with open("/content/hf_token.txt", "r") as f:
            return f.read().strip()
    return None

def extract_model_id(url):
    url = str(url).strip()
    if "huggingface.co" in url:
        match = re.search(r'huggingface\.co/([^/]+/[^/]+)', url)
        if match:
            return match.group(1)
    url = re.sub(r'^https?://', '', url)
    return url

def download_model(model_url, cache_dir, token=None):
    print(f"\n📥 Download: {model_url}")

    if os.path.exists(cache_dir):
        print(f"   Rimozione cache esistente...")
        shutil.rmtree(cache_dir)

    # Pattern per whisper
    dl_patterns = ["*.md", "*.json", "*.pt", "*.pth", "*.bin", "model.safetensors"]

    snapshot_download(
        repo_id=model_url,
        local_dir=cache_dir,
        local_dir_use_symlinks=False,
        resume_download=True,
        allow_patterns=dl_patterns,
        token=token if token else None
    )

    print("✅ Download completato!")
    return cache_dir

def compile_whisper_cpp():
    """Compila whisper.cpp se non esiste"""
    os.chdir("/content")

    if os.path.exists("/content/whisper.cpp/build/bin/quantize"):
        print("✅ whisper.cpp già compilato")
        return True

    print("\n🔨 Compilazione whisper.cpp in corso...")

    if not os.path.exists("whisper.cpp"):
        subprocess.run("git clone https://github.com/ggml-org/whisper.cpp", shell=True)

    os.chdir("whisper.cpp")
    os.makedirs("build", exist_ok=True)
    os.chdir("build")
    subprocess.run("cmake ..", shell=True, capture_output=True)
    subprocess.run("make -j4", shell=True, capture_output=True)
    os.chdir("/content")

    if os.path.exists("/content/whisper.cpp/build/bin/quantize"):
        print("✅ whisper.cpp compilato con successo!")
        return True
    else:
        print("❌ Errore compilazione whisper.cpp")
        return False

def convert_whisper_to_ggml(model_path, output_path):
    """Converti modello Whisper in GGML/GGUF"""
    print(f"\n🔄 Conversione Whisper...")

    os.chdir("/content/whisper.cpp")

    # Cerca il file del modello
    model_file = None
    for f in os.listdir(model_path):
        if f.endswith(".pt") or f.endswith(".pth") or f.endswith(".bin"):
            model_file = os.path.join(model_path, f)
            break

    if not model_file:
        print(f"❌ Nessun file modello trovato in {model_path}")
        return False

    # Usa lo script di conversione appropriato
    if "large" in model_path.lower():
        convert_script = "./models/convert-whisper-to-ggml.py"
    else:
        convert_script = "./models/convert-h5-to-ggml.py"

    cmd = ["python3", convert_script, model_file, output_path]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"❌ Errore conversione: {result.stderr}")
        return False

    print(f"✅ Conversione completata")
    return True

def quantize_whisper(model_path, quant_method, output_path):
    """Quantizza modello Whisper"""
    print(f"\n🔧 Quantizzazione a {quant_method}...")

    os.chdir("/content/whisper.cpp/build")

    cmd = ["./bin/quantize", model_path, output_path, quant_method]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"❌ Errore quantizzazione: {result.stderr}")
        return False

    size = os.path.getsize(output_path) / (1024**3)
    print(f"✅ Quantizzazione completata ({size:.2f} GB)")
    return True

# ============================================
# MAIN ESECUZIONE
# ============================================

print("="*70)
print("🎙️ Quantizzazione Modello Whisper")
print("="*70)

start_time = time.time()

model_url = extract_model_id(whisper_model_url)
model_name = model_url.split('/')[-1]

print(f"📁 Modello: {model_url}")
print(f"🔧 Quantizzazione: {whisper_quant_method}")
print("="*70)

# 1. Compila whisper.cpp se necessario
if not compile_whisper_cpp():
    print("❌ Impossibile procedere senza whisper.cpp")
    sys.exit(1)

# 2. Download modello
token = get_hf_token()
with tempfile.TemporaryDirectory(dir="/content") as download_dir:
    model_cache = os.path.join(download_dir, model_name)
    download_model(model_url, model_cache, token)

    # 3. Converti a GGML
    ggml_path = os.path.join(download_dir, f"{model_name}.bin")
    if not convert_whisper_to_ggml(model_cache, ggml_path):
        sys.exit(1)

    # 4. Quantizza
    if output_filename.strip():
        quantized_name = output_filename.strip()
        if not quantized_name.endswith(".bin"):
            quantized_name += ".bin"
    else:
        quantized_name = f"{model_name}-{whisper_quant_method}.bin"

    quantized_path = os.path.join(download_dir, quantized_name)

    if not quantize_whisper(ggml_path, whisper_quant_method, quantized_path):
        sys.exit(1)

    # 5. Copia in /content/models
    final_path = f"/content/models/{quantized_name}"
    shutil.copy2(quantized_path, final_path)

elapsed = time.time() - start_time

print("\n" + "="*70)
print("✅ QUANTIZZAZIONE COMPLETATA!")
print("="*70)
print(f"📁 File: {final_path}")
print(f"💾 Dimensione: {os.path.getsize(final_path) / (1024**3):.2f} GB")
print(f"⏱️ Tempo: {elapsed/60:.2f} minuti")
print("="*70)

In [ ]:
# @title 🗣️ QUANTIZZA MODELLO VOXTRAL
# @markdown Seleziona modello Voxtral e metodo di quantizzazione

import os
import subprocess
import sys
import time
import shutil
import re
import tempfile
from pathlib import Path
from huggingface_hub import snapshot_download, HfApi

# ============================================
# CONFIGURAZIONE MODELLO
# ============================================

# @markdown ---
# @markdown ### 🤗 Modello Hugging Face
voxtral_model_url = "mistralai/Voxtral-Mini-4B-Realtime-2602" # @param {type:"string"}

# @markdown ---
# @markdown ### 🔧 Metodo di Quantizzazione
voxtral_quant_method = "Q4_K_M" # @param ["Q4_0", "Q4_1", "Q5_0", "Q5_1", "Q8_0", "Q2_K", "Q3_K", "Q4_K", "Q5_K", "Q6_K", "Q4_K_M"]

# @markdown ---
# @markdown ### 📁 Output
output_filename = "" # @param {type:"string"}
# @markdown 💡 Vuoto = automatico: [nome_modello]-[quant].gguf

# ============================================
# FUNZIONI UTILITY
# ============================================

def get_hf_token():
    if os.path.exists("/content/hf_token.txt"):
        with open("/content/hf_token.txt", "r") as f:
            return f.read().strip()
    return None

def extract_model_id(url):
    url = str(url).strip()
    if "huggingface.co" in url:
        match = re.search(r'huggingface\.co/([^/]+/[^/]+)', url)
        if match:
            return match.group(1)
    url = re.sub(r'^https?://', '', url)
    return url

def download_model(model_url, cache_dir, token=None):
    print(f"\n📥 Download: {model_url}")

    if os.path.exists(cache_dir):
        print(f"   Rimozione cache esistente...")
        shutil.rmtree(cache_dir)

    dl_patterns = ["*.md", "*.json", "*.safetensors", "*.bin", "*.model", "tokenizer*"]

    snapshot_download(
        repo_id=model_url,
        local_dir=cache_dir,
        local_dir_use_symlinks=False,
        resume_download=True,
        allow_patterns=dl_patterns,
        token=token if token else None
    )

    print("✅ Download completato!")
    return cache_dir

def compile_voxtral_cpp():
    """Compila voxtral.cpp se non esiste"""
    os.chdir("/content")

    if os.path.exists("/content/voxtral.cpp/build/voxtral-quantize"):
        print("✅ voxtral.cpp già compilato")
        return True

    print("\n🔨 Compilazione voxtral.cpp in corso...")

    if not os.path.exists("voxtral.cpp"):
        subprocess.run("git clone https://github.com/andrijdavid/voxtral.cpp", shell=True)

    os.chdir("voxtral.cpp")
    os.makedirs("build", exist_ok=True)
    os.chdir("build")
    subprocess.run("cmake ..", shell=True, capture_output=True)
    subprocess.run("make -j4", shell=True, capture_output=True)
    os.chdir("/content")

    if os.path.exists("/content/voxtral.cpp/build/voxtral-quantize"):
        print("✅ voxtral.cpp compilato con successo!")
        return True
    else:
        print("❌ Errore compilazione voxtral.cpp")
        return False

def convert_voxtral_to_gguf(model_path, output_path, venv_python):
    """Converti modello Voxtral in GGUF"""
    print(f"\n🔄 Conversione Voxtral a GGUF...")

    os.chdir("/content/voxtral.cpp")
    convert_script = "./convert_hf_to_gguf.py"

    cmd = [venv_python, convert_script, model_path, "--outfile", output_path]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"❌ Errore conversione: {result.stderr}")
        return False

    print(f"✅ Conversione completata")
    return True

def quantize_voxtral(model_path, quant_method, output_path):
    """Quantizza modello Voxtral"""
    print(f"\n🔧 Quantizzazione a {quant_method}...")

    os.chdir("/content/voxtral.cpp/build")

    cmd = ["./voxtral-quantize", model_path, output_path, quant_method]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"❌ Errore quantizzazione: {result.stderr}")
        return False

    size = os.path.getsize(output_path) / (1024**3)
    print(f"✅ Quantizzazione completata ({size:.2f} GB)")
    return True

# ============================================
# MAIN ESECUZIONE
# ============================================

print("="*70)
print("🗣️ Quantizzazione Modello Voxtral")
print("="*70)

start_time = time.time()

# Carica paths
with open("/content/venv_python.txt", "r") as f:
    VENV_PYTHON = f.read().strip()

model_url = extract_model_id(voxtral_model_url)
model_name = model_url.split('/')[-1]

print(f"📁 Modello: {model_url}")
print(f"🔧 Quantizzazione: {voxtral_quant_method}")
print("="*70)

# 1. Compila voxtral.cpp se necessario
if not compile_voxtral_cpp():
    print("❌ Impossibile procedere senza voxtral.cpp")
    sys.exit(1)

# 2. Download modello
token = get_hf_token()
with tempfile.TemporaryDirectory(dir="/content") as download_dir:
    model_cache = os.path.join(download_dir, model_name)
    download_model(model_url, model_cache, token)

    # 3. Converti a GGUF
    gguf_path = os.path.join(download_dir, f"{model_name}.gguf")
    if not convert_voxtral_to_gguf(model_cache, gguf_path, VENV_PYTHON):
        sys.exit(1)

    # 4. Quantizza
    if output_filename.strip():
        quantized_name = output_filename.strip()
        if not quantized_name.endswith(".gguf"):
            quantized_name += ".gguf"
    else:
        quantized_name = f"{model_name}-{voxtral_quant_method}.gguf"

    quantized_path = os.path.join(download_dir, quantized_name)

    if not quantize_voxtral(gguf_path, voxtral_quant_method, quantized_path):
        sys.exit(1)

    # 5. Copia in /content/models
    final_path = f"/content/models/{quantized_name}"
    shutil.copy2(quantized_path, final_path)

elapsed = time.time() - start_time

print("\n" + "="*70)
print("✅ QUANTIZZAZIONE COMPLETATA!")
print("="*70)
print(f"📁 File: {final_path}")
print(f"💾 Dimensione: {os.path.getsize(final_path) / (1024**3):.2f} GB")
print(f"⏱️ Tempo: {elapsed/60:.2f} minuti")
print("="*70)

In [ ]:
# @title 📁 GESTIONE MODELLI QUANTIZZATI
# @markdown Scarica o elimina i modelli salvati

import os
import subprocess
from IPython.display import display, HTML
import ipywidgets as widgets
from pathlib import Path

# ============================================
# FUNZIONI
# ============================================

def list_models():
    """Lista modelli in /content/models"""
    models_dir = "/content/models"
    if not os.path.exists(models_dir):
        os.makedirs(models_dir)
        return []

    models = []
    for f in os.listdir(models_dir):
        fpath = os.path.join(models_dir, f)
        if os.path.isfile(fpath):
            size = os.path.getsize(fpath) / (1024**3)
            models.append({
                "name": f,
                "path": fpath,
                "size_gb": size
            })
    return sorted(models, key=lambda x: x["name"])

def download_model(model_path, model_name):
    """Scarica modello sul PC"""
    from google.colab import files
    print(f"📥 Download {model_name} in corso...")
    files.download(model_path)
    print("✅ Download completato!")

def delete_model(model_path, model_name):
    """Elimina modello"""
    os.remove(model_path)
    print(f"🗑️ Eliminato: {model_name}")
    refresh_list()

def upload_to_hf(model_path, model_name, repo_name, token, is_private=False):
    """Upload modello su Hugging Face"""
    from huggingface_hub import HfApi, create_repo

    print(f"📤 Upload su Hugging Face: {repo_name}")

    try:
        # Crea repository se non esiste
        create_repo(
            repo_id=repo_name,
            token=token,
            private=is_private,
            exist_ok=True
        )

        # Upload file
        api = HfApi()
        api.upload_file(
            path_or_fileobj=model_path,
            path_in_repo=model_name,
            repo_id=repo_name,
            token=token
        )

        print(f"✅ Upload completato! https://huggingface.co/{repo_name}")
        return True
    except Exception as e:
        print(f"❌ Errore upload: {e}")
        return False

def refresh_list():
    """Aggiorna la lista modelli"""
    global models_list, output_area

    models = list_models()

    if not models:
        output_area.clear_output()
        with output_area:
            print("📁 Nessun modello in /content/models")
        return

    html = "<h3>📁 Modelli in /content/models</h3>"
    html += "<table style='width:100%'>"
    html += "<tr><th>Modello</th><th>Dimensione</th><th>Azioni</th></tr>"

    for m in models:
        html += f"""
        <tr>
            <td>{m['name']}</td>
            <td>{m['size_gb']:.2f} GB</td>
            <td>
                <button onclick="download_{m['name'].replace('.', '_')}">📥 Scarica</button>
                <button onclick="delete_{m['name'].replace('.', '_')}">🗑️ Elimina</button>
            </td>
        </tr>
        """

    html += "</table>"

    output_area.clear_output()
    with output_area:
        display(HTML(html))

        # Crea bottoni interattivi
        for m in models:
            btn_download = widgets.Button(description=f"📥 Scarica {m['name']}")
            btn_delete = widgets.Button(description=f"🗑️ Elimina {m['name']}")

            def make_download_callback(path, name):
                return lambda b: download_model(path, name)

            def make_delete_callback(path, name):
                return lambda b: delete_model(path, name)

            btn_download.on_click(make_download_callback(m['path'], m['name']))
            btn_delete.on_click(make_delete_callback(m['path'], m['name']))

            display(widgets.HBox([btn_download, btn_delete]))

# ============================================
# INTERFACCIA
# ============================================

print("="*70)
print("📁 Gestione Modelli Quantizzati")
print("="*70)

# @markdown ### 📤 Upload su Hugging Face (opzionale)
upload_enabled = False # @param {type:"boolean"}
repo_name = "" # @param {type:"string"}
hf_write_token = "" # @param {type:"string"}
is_private = False # @param {type:"boolean"}

# @markdown ---
# @markdown ### 🗑️ Azioni globali
delete_all = False # @param {type:"boolean"}

# ============================================
# ESECUZIONE
# ============================================

# Crea area output
import ipywidgets as widgets
from IPython.display import display

output_area = widgets.Output()

# Mostra modelli
refresh_list()
display(output_area)

# Upload su HF se abilitato
if upload_enabled and hf_write_token and repo_name:
    print("\n📤 Upload modelli su Hugging Face...")
    models = list_models()
    for m in models:
        upload_to_hf(m['path'], m['name'], repo_name, hf_write_token, is_private)

# Elimina tutti se richiesto
if delete_all:
    confirm = input("⚠️ Sei sicuro di voler eliminare TUTTI i modelli? (si/no): ")
    if confirm.lower() == "si":
        models = list_models()
        for m in models:
            os.remove(m['path'])
        print(f"🗑️ Eliminati {len(models)} modelli")
        refresh_list()
    else:
        print("Operazione annullata")

print("\n✅ Operazioni completate!")